In [ ]:
# !pip install pandas openpyxl jieba wordcloud matplotlib seaborn networkx numpy pyvis

数据读取+文本预处理

In [ ]:
import pandas as pd
import jieba
import re
import itertools
import random
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import networkx as nx
from pyvis.network import Network
from collections import Counter, defaultdict

# ====================== 配置项 ======================
# 数据路径（替换为实际路径）
DATA_PATH = r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\合并数据（不含学科）.xlsx"
# 停用词表路径（若无文件，可使用内置停用词）
STOPWORDS_PATH = "stopwords.txt"
# 词云图保存路径
WORDCLOUD_SAVE_PATH = r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\TOP30关键词词云图.png"
# 共现网络图保存路径
COOC_NET_SAVE_PATH = r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\关键词共现网络图.png"

# 设置中文字体（解决词云/图表中文乱码）
plt.rcParams["font.family"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

# ====================== 1. 读取数据 ======================
df = pd.read_excel(DATA_PATH)
# 提取项目名称列，去空值
project_names = df["项目名称"].dropna().tolist()
print(f"有效项目名称数量：{len(project_names)}")

# ====================== 2. 文本预处理（分词+去停用词） ======================
# 加载停用词（优先读取文件，无则用内置）
def load_stopwords():
    try:
        with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
            stopwords = [line.strip() for line in f if line.strip()]
    except:
        # 内置基础停用词（可根据需求补充）
        stopwords = [
            "研究", "基于", "分析", "探讨", "探析", "试论", "浅谈", "关于", "以", "为", "的", "与", "和",
            "对", "及", "等", "从", "论", "略论", "基于", "面向", "一种", "方法", "路径", "机制", "模式",
            "体系", "构建", "实践", "应用", "探究", "考察", "梳理", "阐释", "解读", "思考", "刍议", "初探",
            "及其",
        ]
    return stopwords

stopwords = load_stopwords()

# 分词函数（去除非中文字符，仅保留名词/动词核心词）
def cut_project_name(name):
    # 去除特殊字符、数字、字母
    name_clean = re.sub(r"[^\u4e00-\u9fa5]", "", str(name))
    # 分词
    words = jieba.lcut(name_clean)
    # 过滤停用词、单字
    words_filtered = [w for w in words if w not in stopwords and len(w) > 1]
    return words_filtered

# 对所有项目名称分词
all_words = []
word_list_per_project = []  # 每个项目的分词结果（用于共现分析）
for name in project_names:
    words = cut_project_name(name)
    all_words.extend(words)
    word_list_per_project.append(words)

# 统计TOP30关键词
word_counter = Counter(all_words)
top30_words = word_counter.most_common(30)
print("\nTOP30关键词：")
for idx, (word, count) in enumerate(top30_words, 1):
    print(f"{idx}. {word} - {count}次")

词云图展示

In [ ]:
# ====================== 3. TOP30关键词词云图（颜色多样化） ======================
wordcloud_data = dict(top30_words)
# 生成词云（使用随机彩色配色）
wc = WordCloud(
    font_path="msyh.ttc",  # 系统中文字体路径
    width=1000,
    height=600,
    background_color="white",
    max_words=30,
    font_step=1,
    random_state=42,
    collocations=False,
    # 关键修改：使用随机彩色（或自定义颜色函数）
    color_func=lambda *args, **kwargs: np.random.choice([
        "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57",
        "#FF9FF3", "#54A0FF", "#5F27CD", "#FF9F43", "#10AC84"
    ])  # 自定义10种鲜明颜色
)
wc.generate_from_frequencies(wordcloud_data)

# 绘制并保存
plt.figure(figsize=(12, 8))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("2020-2025年项目名称TOP30关键词词云图", fontsize=16, pad=20)
plt.tight_layout()
plt.savefig(WORDCLOUD_SAVE_PATH, dpi=300, bbox_inches="tight")
plt.show()
print(f"\n✅ 彩色词云图已保存至：{WORDCLOUD_SAVE_PATH}")

关键词共现网络图展示

In [ ]:
# TOP30 关键词集合（用于过滤，只保留这些高频词的共现）
top30_set = set(word for word, count in top30_words)

# 统计共现次数
cooc_dict = defaultdict(int)

for words in word_list_per_project:
    # 只保留出现在TOP30中的词
    filtered_words = [w for w in words if w in top30_set]
    if len(filtered_words) < 2:
        continue
    # 所有两两组合（无向）
    for w1, w2 in itertools.combinations(filtered_words, 2):
        if w1 == w2:
            continue
        # 统一排序，保证 (A,B) 和 (B,A) 算同一边
        pair = tuple(sorted([w1, w2]))
        cooc_dict[pair] += 1

# 构建 NetworkX 图
G = nx.Graph()

# 添加节点（节点大小与词频成正比）
for word, freq in top30_words:
    G.add_node(word, size=freq, title=f"{word} ({freq}次)", label=word)

# 添加边（只保留共现次数 >= 阈值的边）
min_cooc_threshold = 30  # 可根据实际情况调小/调大，建议20~50之间

for (w1, w2), weight in cooc_dict.items():
    if weight >= min_cooc_threshold:
        G.add_edge(w1, w2, weight=weight, title=f"共现 {weight} 次")

print(f"共现网络节点数：{G.number_of_nodes()}")
print(f"共现网络边数（阈值≥{min_cooc_threshold}）：{G.number_of_edges()}")

# ====================== 4.1 静态图 ======================
plt.figure(figsize=(14, 10))

# 节点大小映射
node_sizes = [G.nodes[n]['size'] * 15 for n in G.nodes()]  # 放大系数，可调整

# 边宽度映射
edge_weights = [G.edges[e]['weight'] for e in G.edges()]
edge_widths = [w / 10 for w in edge_weights]  # 调整显示粗细

pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)  # 布局更美观

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color="#FF9F43", alpha=0.9)
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.6, edge_color="#54A0FF")
nx.draw_networkx_labels(G, pos, font_size=12, font_family="SimHei")

plt.title("2020-2025年项目名称TOP30关键词共现网络图（静态版）", fontsize=18, pad=20)
plt.axis("off")
plt.tight_layout()

# 保存静态图
STATIC_NET_SAVE_PATH = r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\关键词共现网络图_静态.png"
plt.savefig(STATIC_NET_SAVE_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ 静态共现网络图已保存至：{STATIC_NET_SAVE_PATH}")

# ====================== 4.2交互式图（适配pyvis） ======================

from pyvis.network import Network

net = Network(
    height="900px",
    width="100%",
    bgcolor="#ffffff",
    font_color="black",
    notebook=True,
    directed=False,
    cdn_resources="in_line"
)

net.from_nx(G)

# 基础物理参数
net.barnes_hut(
    gravity=-6000,
    central_gravity=0.03,
    spring_length=350,
    spring_strength=0.03,
    damping=0.92,
    overlap=1
)

# 新版 pyvis 用直接赋值方式设置 options
net.options = {
    "physics": {
        "enabled": True,
        "barnesHut": {
            "gravitationalConstant": -6000,
            "centralGravity": 0.03,
            "springLength": 350,
            "springStrength": 0.03,
            "damping": 0.92,
            "overlap": 1
        },
        "stabilization": {
            "enabled": True,
            "iterations": 6000,       # 长动画
            "updateInterval": 30,
            "fit": True
        },
        "timestep": 0.5,
        "adaptiveTimestep": True,
        "maxVelocity": 50          # 防止节点飞太远
    },
    "edges": {
        "smooth": {
            "type": "cubicBezier",
            "roundness": 0.4
        }
    },
    "layout": {
        "improvedLayout": True
    },
    "interaction": {
        "dragView": True,
        "zoomView": True,
        "hover": True
    }
}

# 节点和边美化
max_freq = max(freq for _, freq in top30_words)
min_freq = min(freq for _, freq in top30_words)

for node in net.nodes:
    word = node["label"]
    freq = next(f for w, f in top30_words if w == word)
    size = 30 + (freq - min_freq) / (max_freq - min_freq + 1e-5) * 90
    node["size"] = size
    node["color"] = {
        "background": "#FF6B35",
        "border": "#D35400",
        "highlight": {"background": "#FF4500", "border": "#C0392B"}
    }
    node["font"] = {
        "size": 24,
        "face": "SimHei",
        "color": "white",
        "strokeWidth": 3,
        "strokeColor": "black"
    }
    node["title"] = f"{word}<br>出现次数：{freq}次"

for edge in net.edges:
    weight = edge.get("weight", 1)
    edge["width"] = 1 + (weight - min_cooc_threshold) / 100 * 9
    edge["color"] = {"color": "#4A90E2", "highlight": "#2B6ECC", "opacity": 0.8}
    edge["title"] = f"共现 {weight} 次"

# 手动保存 + 插入自动 fit 脚本（防止跑出屏幕）
INTERACTIVE_NET_SAVE_PATH = r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\关键词共现网络图_交互式.html"

html_content = net.generate_html()

# 插入稳定后自动缩放全图的脚本
fit_script = """
<script type="text/javascript">
  network.on("stabilized", function () {
    network.fit({
      animation: {
        duration: 1000,
        easingFunction: "easeInOutQuad"
      }
    });
  });
</script>
"""

html_content = html_content.replace("</body>", fit_script + "</body>")

with open(INTERACTIVE_NET_SAVE_PATH, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"✅ 交互式共现网络图已生成！")
print(f"   路径：{INTERACTIVE_NET_SAVE_PATH}")